# HomeTown AR: ALL models in one notebook (furniture 3D + room understanding + redesign)
Runtime > Change runtime type > **T4 GPU**. Then follow the steps **in order**. Every step says what it makes.

| Step | What it does | Output |
|---|---|---|
| 1 | Installs everything (run once, then **Runtime > Restart session**) | - |
| 2 | Google Drive, `products.json`, Hugging Face token | - |
| 3 | Writes the worker script | - |
| 4 | Runs the 3D models, one after another, on the same clean SAM 2 cut-outs | `<id>.glb` in a folder per model |
| 5 | Zips each model separately | `models_hunyuan_mv.zip`, `models_hunyuan_mini.zip`, `models_triposr.zip`, `models_sf3d.zip` |
| 6 | Room photos: Depth Anything V2 + Metric3D depth | `<room>__depth_*.png` |
| 7 | Room photos: Grounding DINO + SAM 2 (sofa, bed, wall, window, door ...) | `<room>__objects.json` + overlay |
| 8 | Room redesign concepts: SDXL + depth ControlNet, CLIP style check | `<room>__redesign_*.png` |
| 9 | Zips the room outputs | `room_outputs.zip` |

**The 3D models** (all use the same SAM 2 + background-removed cut-outs, so they are comparable):
`hunyuan_mv` (multi-photo, best quality, slowest) · `hunyuan_mini` (one photo, faster) · `triposr` (one photo, very fast) · `sf3d` (Stable Fast 3D, one photo, keeps its texture, needs a free Hugging Face licence).
Different models give different results; compare them and keep the best per product. Output files are named `<product-id>__<model>.glb`.

Everything is saved to Google Drive as it goes, so a stopped session just resumes: run steps 2, 3, 4 again, finished products are skipped.
Each 3D model runs in its own short-lived process, so one model that fails to install or crashes does not stop the others.

## Step 1: install (then Runtime > Restart session)

In [ ]:
!nvidia-smi -L
# Hunyuan3D-2 (hunyuan_mv, hunyuan_mini)
!rm -rf /content/Hunyuan3D-2
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2 /content/Hunyuan3D-2
%cd /content/Hunyuan3D-2
!pip -q install -r requirements.txt
!pip -q install -e .
# shared tools: background removal, mesh clean-up, SAM 2 (segmentation)
!pip -q install rembg onnxruntime-gpu scipy scikit-image opencv-python-headless pymeshlab psutil
!pip -q install --no-deps git+https://github.com/facebookresearch/sam2.git
!pip -q install hydra-core iopath
# TripoSR (triposr)
%cd /content
!rm -rf /content/TripoSR
!git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR /content/TripoSR
!pip -q install omegaconf einops xatlas moderngl imageio[ffmpeg]
!pip -q install git+https://github.com/tatsy/torchmcubes.git || echo "TRIPOSR: torchmcubes failed, triposr will be skipped"
# Stable Fast 3D (sf3d). Its own requirements pin old versions, so install the pieces it needs without the pins
!rm -rf /content/stable-fast-3d
!git clone --depth 1 https://github.com/Stability-AI/stable-fast-3d /content/stable-fast-3d
!pip -q install jaxtyping open_clip_torch gpytoolbox pynanoinstantmeshes safetensors
!pip -q install /content/stable-fast-3d/texture_baker /content/stable-fast-3d/uv_unwrapper || echo "SF3D: native extensions failed, sf3d will be skipped"
# room understanding + redesign (steps 6 to 8)
!pip -q install diffusers accelerate
# Colab's newest transformers renames layers the Hunyuan3D-2 weights expect: use the version they were made for
!pip -q install "transformers==4.46.3" "huggingface-hub<1.0"
!pip cache purge
# NOW: Runtime > Restart session, then run step 2 (not this one again)

## Step 2: Drive, products.json, Hugging Face token
- `products.json` comes from your PC: `node scripts/colab-batch.mjs export --all --redo` (or `--next 20` for a quick test) writes `models/colab/products.json`.
- The Hugging Face token is only needed for **sf3d** (Stable Fast 3D is gated): on huggingface.co open `stabilityai/stable-fast-3d`, accept the licence, create a read token at Settings > Access Tokens. Leave it blank to skip sf3d.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os, json, getpass
OUT = '/content/drive/MyDrive/hometown_models_all'
os.makedirs(OUT, exist_ok=True)
files.upload()                      # choose products.json
products = json.load(open('products.json'))
HF_TOKEN = getpass.getpass('Hugging Face token for sf3d (blank = skip): ').strip()
if HF_TOKEN:
    from huggingface_hub import login; login(HF_TOKEN)
def done(model): return len([f for f in os.listdir(f'{OUT}/{model}') if f.endswith('.glb') or f.endswith('.skip')]) if os.path.isdir(f'{OUT}/{model}') else 0
print(len(products), 'products |', {m: done(m) for m in ['hunyuan_mv', 'hunyuan_mini', 'triposr', 'sf3d']}, 'already done')

## Step 3: write the worker script (nothing to change)

In [ ]:
%%writefile /content/worker.py
# Does up to BATCH unfinished products with ONE chosen 3D model, then exits (frees all RAM).
# usage: worker.py products.json OUT BATCH MODEL     MODEL = hunyuan_mv | hunyuan_mini | triposr | sf3d
# results: OUT/<MODEL>/<id>.glb and OUT/<MODEL>/report.json
import os, sys, io, gc, json, time, ctypes
os.environ['OMP_NUM_THREADS'] = '2'; os.environ['MKL_NUM_THREADS'] = '2'; os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np, requests, cv2, torch, trimesh, psutil
from PIL import Image
from scipy import ndimage as ndi
sys.path.insert(0, '/content/Hunyuan3D-2')

PRODUCTS, OUT, BATCH, MODEL = sys.argv[1], sys.argv[2], int(sys.argv[3]), sys.argv[4]
assert MODEL in ('hunyuan_mv', 'hunyuan_mini', 'triposr', 'sf3d'), MODEL
OUT = f'{OUT}/{MODEL}'; os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, '/content/TripoSR'); sys.path.insert(0, '/content/stable-fast-3d')
SEEDS = [12355, 7] if MODEL.startswith('hunyuan') else [0]   # candidates per product (TripoSR / SF3D are deterministic: 1)
OCTREE = 320                  # lower = less RAM. 384/512 = smoother but heavier
RAM_STOP = 85                 # exit cleanly above this % RAM
PAUSE = 2
USE_SAM2 = True               # SAM 2 refines the rembg cut-out edge (falls back to rembg if not installed)
MIN_FIT = 0.55                # photo-match below this is flagged in report.json
products = json.load(open(PRODUCTS))
finished = lambda p: os.path.exists(f"{OUT}/{p['id']}.glb") or os.path.exists(f"{OUT}/{p['id']}.skip")
todo = [p for p in products if not finished(p)][:BATCH]
if not todo:
    print('ALL_DONE'); sys.exit(0)

def ram(): return psutil.virtual_memory().percent
def cleanup():
    gc.collect(); torch.cuda.empty_cache()
    try: ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception: pass

from hy3dgen.rembg import BackgroundRemover
from transformers import pipeline as hf_pipeline
model3d = None
if MODEL in ('hunyuan_mv', 'hunyuan_mini'):
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
    if MODEL == 'hunyuan_mv':   # multi-view model: also accepts a single front photo
        model3d = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mv', subfolder='hunyuan3d-dit-v2-mv', variant='fp16')
    else:
        model3d = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini', variant='fp16')
elif MODEL == 'triposr':
    from tsr.system import TSR
    model3d = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
    model3d.renderer.set_chunk_size(8192); model3d.to('cuda')
elif MODEL == 'sf3d':   # gated on Hugging Face: accept the licence and log in first (notebook step 2)
    from sf3d.system import SF3D
    model3d = SF3D.from_pretrained('stabilityai/stable-fast-3d', config_name='config.yaml', weight_name='model.safetensors')
    model3d.to('cuda'); model3d.eval()
rembg_ = BackgroundRemover()
clip = hf_pipeline('zero-shot-image-classification', model='openai/clip-vit-base-patch32', device=0, torch_dtype=torch.float16)
sam = None
if USE_SAM2:
    try:
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        sam = SAM2ImagePredictor.from_pretrained('facebook/sam2.1-hiera-small')
        print('SAM 2 loaded', flush=True)
    except Exception as e:
        print('SAM 2 unavailable, using rembg only:', str(e)[:120], flush=True)
cleanup(); print('loaded, RAM %.0f%%' % ram(), flush=True)

LABELS = {
    'front': 'a product photo of furniture seen straight from the front on a plain background',
    'side': 'a product photo of furniture seen straight from the side on a plain background',
    'back': 'a product photo of furniture seen from the back on a plain background',
    'angled': 'a product photo of furniture seen from a three-quarter angle on a plain background',
    'room': 'furniture in a decorated room interior, a lifestyle photo',
    'detail': 'a close-up detail of furniture material or hardware',
}
INV = {v: k for k, v in LABELS.items()}

def sam_refine(rgb, m):
    """SAM 2 prompted with the rembg box: crisper legs/arms. Only accepted if it agrees with rembg (IoU > 0.8)."""
    if sam is None or not m.any(): return m
    try:
        ys, xs = np.where(m)
        box = np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=np.float32)
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
            sam.set_image(np.array(rgb.convert('RGB')))
            masks, scores, _ = sam.predict(box=box, multimask_output=False)
        r = masks[0].astype(bool)
        lab, n = ndi.label(r)
        if n > 1:
            sizes = ndi.sum(r, lab, range(1, n + 1)); r = lab == (1 + int(np.argmax(sizes)))
        r = ndi.binary_fill_holes(r)
        iou = (r & m).sum() / max((r | m).sum(), 1)
        return r if iou > 0.8 else m
    except Exception:
        return m


def clean_cutout(rgb, size=1024, pad=0.08):
    """Remove background, keep the largest object, fill holes, soften the edge, centre and pad on a square canvas."""
    cut = rembg_(rgb.convert('RGB')).convert('RGBA')
    a = np.array(cut.split()[-1]).astype(np.float32) / 255
    m = a > 0.5
    lab, n = ndi.label(m)
    if n > 1:
        sizes = ndi.sum(m, lab, range(1, n + 1)); m = lab == (1 + int(np.argmax(sizes)))
    m = ndi.binary_fill_holes(m)
    m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8)).astype(bool)
    m = sam_refine(rgb, m)
    soft = cv2.GaussianBlur(m.astype(np.float32), (0, 0), 1.2)
    ys, xs = np.where(m)
    if len(ys) == 0: return None, 0.0
    arr = np.array(cut); arr[..., 3] = (soft * 255).astype(np.uint8); arr[~m, :3] = 255
    crop = Image.fromarray(arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1])
    s = int(max(crop.size) * (1 + 2 * pad))
    canvas = Image.new('RGBA', (s, s), (255, 255, 255, 0))
    canvas.paste(crop, ((s - crop.width) // 2, (s - crop.height) // 2))
    canvas = canvas.resize((size, size), Image.LANCZOS)
    touch = np.concatenate([m[0], m[-1], m[:, 0], m[:, -1]]).mean(); cover = m.mean()
    return canvas, (0.0 if (cover < 0.10 or cover > 0.93) else float(1 - touch))

def sharpness(rgb):
    g = cv2.cvtColor(np.array(rgb), cv2.COLOR_RGB2GRAY)
    return float(min(cv2.Laplacian(g, cv2.CV_64F).var() / 300, 1.0))

def analyse(photos):
    items = []
    for u in photos:
        try:
            im = Image.open(io.BytesIO(requests.get(u, timeout=60).content)).convert('RGB'); im.thumbnail((1200, 1200))
        except Exception:
            continue
        cut, q = clean_cutout(im)
        if cut is None: continue
        r = clip(im, candidate_labels=list(LABELS.values()))
        cls = {INV[x['label']]: x['score'] for x in r}
        items.append({'url': u, 'cut': cut, 'cls': cls, 'top': max(cls, key=cls.get), 'quality': q * (0.5 + 0.5 * sharpness(im))})
    return items

def pick_views(items):
    good = [i for i in items if i['quality'] > 0.4 and i['top'] not in ('room', 'detail')]
    def best(k, pool):
        pool = [i for i in pool if i['top'] == k]
        return max(pool, key=lambda i: i['cls'][k] * i['quality']) if pool else None
    front = best('front', good) or best('angled', good)
    if front is None and items: front = max(items, key=lambda i: i['quality'])
    rest = [i for i in good if i is not front]
    return front, best('side', rest), best('back', rest)

import pymeshlab
def postprocess(mesh, target_tris=40000):
    ms = pymeshlab.MeshSet()
    ms.add_mesh(pymeshlab.Mesh(np.asarray(mesh.vertices, dtype=np.float64), np.asarray(mesh.faces, dtype=np.int32)))
    ms.meshing_remove_duplicate_vertices(); ms.meshing_remove_unreferenced_vertices()
    ms.meshing_remove_connected_component_by_diameter(mincomponentdiag=pymeshlab.PercentageValue(8))
    ms.meshing_repair_non_manifold_edges(); ms.meshing_close_holes(maxholesize=60)
    ms.apply_coord_taubin_smoothing(lambda_=0.5, mu=-0.53, stepsmoothnum=12)
    if ms.current_mesh().face_number() > target_tris:
        ms.meshing_decimation_quadric_edge_collapse(targetfacenum=target_tris, preserveboundary=True, preservenormal=True, qualitythr=0.6)
    ms.apply_coord_taubin_smoothing(lambda_=0.5, mu=-0.53, stepsmoothnum=4)
    m = ms.current_mesh()
    out = trimesh.Trimesh(m.vertex_matrix(), m.face_matrix(), process=True); out.fix_normals()
    ms.clear(); return out

def score(mesh, d):
    """Higher is better: one solid piece, watertight, proportions close to the listed W x D x H."""
    parts = mesh.split(only_watertight=False)
    solid = max((len(p.faces) for p in parts), default=1) / max(len(mesh.faces), 1)
    ext = np.sort(mesh.extents); want = np.sort(np.array([d['w'], d['d'], d['h']], dtype=float))
    err = float(np.abs(np.log((ext / ext.max() + 1e-6) / (want / want.max()))).mean())
    return solid + (0.5 if mesh.is_watertight else 0) - 1.5 * err

def silhouette(mesh, axes, flip, size=256):
    """Orthographic silhouette of the mesh along one axis pair, as a size x size mask."""
    v = mesh.vertices[:, axes].astype(np.float64) * np.array(flip)
    lo, hi = v.min(0), v.max(0); sc = (size - 6) / max((hi - lo).max(), 1e-9)
    pts = ((v - lo) * sc + 3 + (size - 6 - (hi - lo) * sc) / 2).astype(np.int32)
    img = np.zeros((size, size), np.uint8)
    tri = pts[mesh.faces]
    for t in tri[:: max(1, len(tri) // 30000)]: cv2.fillConvexPoly(img, t, 1)
    return img.astype(bool)

def _norm_mask(m, size=256):
    """Crop to the object, scale so the longest side fits (aspect kept), centre on a size x size canvas."""
    ys, xs = np.where(m)
    if len(ys) == 0: return None
    c = m[ys.min():ys.max() + 1, xs.min():xs.max() + 1].astype(np.uint8) * 255
    k = (size - 6) / max(c.shape)
    c = cv2.resize(c, (max(1, round(c.shape[1] * k)), max(1, round(c.shape[0] * k)))) > 127
    out = np.zeros((size, size), bool)
    y0, x0 = (size - c.shape[0]) // 2, (size - c.shape[1]) // 2
    out[y0:y0 + c.shape[0], x0:x0 + c.shape[1]] = c
    return out

def photo_fit(mesh, cut):
    """0..1: best overlap (IoU, aspect kept) between the photo cut-out and any axis-aligned silhouette of the mesh, so it does not matter which way the model faces."""
    ref = _norm_mask(np.array(cut.split()[-1].resize((256, 256))) > 127)
    if ref is None: return 0.0
    best = 0.0
    for ax in ([0, 1], [0, 2], [2, 1]):
        for fl in ([1, 1], [-1, 1], [1, -1], [-1, -1]):
            mm = _norm_mask(silhouette(mesh, ax, fl))
            if mm is not None: best = max(best, (mm & ref).sum() / max((mm | ref).sum(), 1))
    return float(best)

def tint(mesh, hexcol):
    c = tuple(int(hexcol.lstrip('#')[i:i + 2], 16) for i in (0, 2, 4)) + (255,)
    mesh.visual = trimesh.visual.ColorVisuals(mesh=mesh, vertex_colors=np.tile(np.array(c, dtype=np.uint8), (len(mesh.vertices), 1)))
    return mesh

def on_grey(cut, ratio=0.85):
    """RGBA cut-out composited on mid-grey, the input TripoSR was trained on."""
    arr = np.array(cut).astype(np.float32) / 255
    out = arr[..., :3] * arr[..., 3:] + 0.5 * (1 - arr[..., 3:])
    return Image.fromarray((out * 255).astype(np.uint8))

@torch.inference_mode()
def generate(front, side, back, seed, octree):
    if MODEL == 'hunyuan_mv':
        v = {'front': front['cut']}
        if side is not None: v['left'] = side['cut']
        if back is not None: v['back'] = back['cut']
        kw = dict(num_inference_steps=50, guidance_scale=7.5, octree_resolution=octree, num_chunks=10000, generator=torch.manual_seed(seed))
        return model3d(image=v, **kw)[0], ('multi-view' if len(v) > 1 else 'single-view')
    if MODEL == 'hunyuan_mini':
        kw = dict(num_inference_steps=50, guidance_scale=7.5, octree_resolution=octree, num_chunks=10000, generator=torch.manual_seed(seed))
        return model3d(image=front['cut'], **kw)[0], 'single-view'
    if MODEL == 'triposr':
        codes = model3d([on_grey(front['cut'])], device='cuda')
        return model3d.extract_mesh(codes, True, resolution=256)[0], 'single-view'
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):      # sf3d: already UV-textured and light
        mesh, _ = model3d.run_image(front['cut'].resize((512, 512)), bake_resolution=1024, remesh='none')
    return mesh, 'single-view'

rep_path = f'{OUT}/report.json'
report = json.load(open(rep_path)) if os.path.exists(rep_path) else {}
for p in todo:
    if ram() > RAM_STOP:
        print('RAM %.0f%%, exiting to free memory' % ram(), flush=True); break
    t0 = time.time()
    try:
        items = analyse(p.get('photos') or [p['photo']])
        if not items:
            print('no usable photos', p['id'], flush=True); open(f"{OUT}/{p['id']}.skip", 'w').write('x'); continue
        front, side, back = pick_views(items)
        best, best_s, mode = None, -9, ''
        for sd in SEEDS:
            try: raw, mode = generate(front, side, back, sd, OCTREE)
            except torch.cuda.OutOfMemoryError: cleanup(); raw, mode = generate(front, side, back, sd, 256)
            m = raw if MODEL == 'sf3d' else postprocess(raw)
            del raw
            s = score(m, p['dimensionsCm']) + 1.5 * photo_fit(m, front['cut'])
            if s > best_s: best, best_s = m, s
            cleanup(); time.sleep(1)
        if MODEL != 'sf3d': best = tint(best, p.get('dominantHex', '#b8a58f'))   # sf3d keeps its baked texture
        fit = photo_fit(best, front['cut'])
        best.export(f"{OUT}/{p['id']}.glb")
        report[p['id']] = {'mode': mode, 'score': round(best_s, 3), 'tris': int(len(best.faces)), 'watertight': bool(best.is_watertight), 'photoFit': round(fit, 3), 'review': bool(fit < MIN_FIT), 'views': [i['top'] for i in items]}
        json.dump(report, open(rep_path, 'w'), indent=1)
        print(f"{p['id']} {mode} score={best_s:.2f} tris={len(best.faces)} RAM {ram():.0f}% {int(time.time() - t0)}s", flush=True)
    except Exception as e:
        print('FAILED', p['id'], str(e)[:160], flush=True)
    cleanup(); time.sleep(PAUSE)

## Step 4: run the 3D models
Edit `MODELS` to run only some (for example `['triposr']` for a fast first look). Re-run this cell whenever it stops; it resumes.
Order is fastest first, so you get results early. A model that fails 5 times in a row is skipped and the next one starts.

In [ ]:
import subprocess, sys, time
MODELS = ['triposr', 'hunyuan_mini', 'hunyuan_mv', 'sf3d']
if not HF_TOKEN and 'sf3d' in MODELS: MODELS.remove('sf3d'); print('sf3d skipped (no Hugging Face token)')
BATCH = 8                       # products per worker process (lower = even safer)
for MODEL in MODELS:
    print('=== ', MODEL, ' ===')
    fails = 0
    while True:
        before = done(MODEL)
        r = subprocess.run([sys.executable, '/content/worker.py', 'products.json', OUT, str(BATCH), MODEL], capture_output=True, text=True)
        print(r.stdout[-1200:])
        if 'ALL_DONE' in r.stdout:
            print(MODEL, 'finished'); break
        if r.returncode != 0 or done(MODEL) == before:
            fails += 1; print('worker exited (code %s), restarting. %s' % (r.returncode, r.stderr[-400:]))
            if fails >= 5:
                print(MODEL, ': 5 failures in a row, skipping this model. Last error above.'); break
            time.sleep(10)
        else:
            fails = 0
        print(MODEL, 'progress:', done(MODEL), '/', len(products))

## Step 5: zip each model separately
Makes `models_<model>.zip` (files named `<id>__<model>.glb`) and `report_<model>.json` in your Drive folder `hometown_models_all/zips`, and downloads them.
If the browser blocks multiple downloads, click Allow, or take them from Drive.

In [ ]:
import zipfile
os.makedirs(f'{OUT}/zips', exist_ok=True)
for m in ['hunyuan_mv', 'hunyuan_mini', 'triposr', 'sf3d']:
    d = f'{OUT}/{m}'
    glbs = sorted(f for f in os.listdir(d) if f.endswith('.glb')) if os.path.isdir(d) else []
    if not glbs: print(m, ': nothing to zip'); continue
    z = f'{OUT}/zips/models_{m}.zip'
    with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in glbs: zf.write(f'{d}/{f}', f.replace('.glb', f'__{m}.glb'))
    print(m, len(glbs), 'models,', os.path.getsize(z) // 1_000_000, 'MB')
    files.download(z)
    if os.path.exists(f'{d}/report.json'): files.download(f'{d}/report.json')

## Step 6: room photos, depth (Depth Anything V2 + Metric3D)
Upload one or more photos of a room. Depth from ONE ordinary photo is relative, not exact: use it for rough structure, and use a real room scan or measured dimensions for true sizes.

In [ ]:
from PIL import Image
import numpy as np, torch, gc
ROOM_OUT = f'{OUT}/rooms'; os.makedirs(ROOM_OUT, exist_ok=True)
up = files.upload()                  # choose room photos (jpg / png)
rooms = {os.path.splitext(n)[0]: n for n in up}
from transformers import pipeline
depth_pipe = pipeline('depth-estimation', model='depth-anything/Depth-Anything-V2-Base-hf', device=0)
for name, fn in rooms.items():
    im = Image.open(fn).convert('RGB')
    d = depth_pipe(im)['depth']; d.save(f'{ROOM_OUT}/{name}__depth_anything_v2.png')
    print(name, 'depth anything v2 ok')
del depth_pipe; gc.collect(); torch.cuda.empty_cache()

# Metric3D (metric depth, approximate: the focal length below is a guess for a phone camera)
try:
    m3d = torch.hub.load('yvanyin/metric3d', 'metric3d_vit_small', pretrain=True).cuda().eval()
    FX = 1000.0
    for name, fn in rooms.items():
        rgb = np.array(Image.open(fn).convert('RGB')); h, w = rgb.shape[:2]
        s = min(616 / h, 1064 / w); rgb = np.array(Image.fromarray(rgb).resize((int(w * s), int(h * s))))
        ph, pw = 616 - rgb.shape[0], 1064 - rgb.shape[1]
        pad = np.pad(rgb, ((ph // 2, ph - ph // 2), (pw // 2, pw - pw // 2), (0, 0)), constant_values=114)
        mean = torch.tensor([123.675, 116.28, 103.53]).view(3, 1, 1).cuda(); std = torch.tensor([58.395, 57.12, 57.375]).view(3, 1, 1).cuda()
        x = ((torch.from_numpy(pad.transpose(2, 0, 1)).float().cuda() - mean) / std)[None]
        with torch.no_grad(): pred, _, _ = m3d.inference({'input': x})
        pred = pred.squeeze()[ph // 2: 616 - (ph - ph // 2), pw // 2: 1064 - (pw - pw // 2)] * (FX * s / 1000.0)
        arr = pred.cpu().numpy(); np.save(f'{ROOM_OUT}/{name}__metric3d_metres.npy', arr)
        Image.fromarray((255 * (arr - arr.min()) / (np.ptp(arr) + 1e-6)).astype(np.uint8)).save(f'{ROOM_OUT}/{name}__depth_metric3d.png')
        print(name, 'metric3d ok, median depth %.1f m' % float(np.median(arr)))
    del m3d; gc.collect(); torch.cuda.empty_cache()
except Exception as e:
    print('Metric3D skipped:', str(e)[:200])

## Step 7: what is in the room (Grounding DINO + SAM 2)

In [ ]:
import json
from sam2.sam2_image_predictor import SAM2ImagePredictor
LABELS = ['sofa', 'bed', 'chair', 'table', 'wardrobe', 'wall', 'window', 'door', 'rug', 'lamp', 'tv']
det = pipeline('zero-shot-object-detection', model='IDEA-Research/grounding-dino-tiny', device=0)
sam = SAM2ImagePredictor.from_pretrained('facebook/sam2.1-hiera-small')
rng = np.random.default_rng(0)
for name, fn in rooms.items():
    im = Image.open(fn).convert('RGB'); arr = np.array(im)
    res = [r for r in det(im, candidate_labels=LABELS, threshold=0.3)]
    overlay = arr.astype(np.float32).copy(); found = []
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        sam.set_image(arr)
        for r in res:
            b = r['box']; box = np.array([b['xmin'], b['ymin'], b['xmax'], b['ymax']], dtype=np.float32)
            mask = sam.predict(box=box, multimask_output=False)[0][0].astype(bool)
            overlay[mask] = 0.55 * overlay[mask] + 0.45 * rng.integers(40, 255, 3)
            found.append({'label': r['label'], 'score': round(float(r['score']), 3), 'box': [round(float(v)) for v in box], 'area_fraction': round(float(mask.mean()), 4)})
    Image.fromarray(overlay.astype(np.uint8)).save(f'{ROOM_OUT}/{name}__objects.png')
    json.dump(found, open(f'{ROOM_OUT}/{name}__objects.json', 'w'), indent=1)
    print(name, [f['label'] for f in found])
del det, sam; gc.collect(); torch.cuda.empty_cache()

## Step 8: redesign concepts (SDXL + depth ControlNet, checked with CLIP)
Edit `STYLES`. These are **design pictures only**, not an accurate 3D room: keep them separate from the true-to-size AR scene.
(FLUX and an LLM design assistant are not run here: FLUX does not fit a free T4, and the assistant belongs in the app, not in Colab.)

In [ ]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline, AutoencoderKL
STYLES = ['minimalist Scandinavian living room, light oak furniture, warm neutral palette',
          'warm modern Indian home, walnut wood furniture, rust and gold accents']
controlnet = ControlNetModel.from_pretrained('diffusers/controlnet-depth-sdxl-1.0', torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', controlnet=controlnet, vae=vae, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
made = []
for name in rooms:
    dimg = Image.open(f'{ROOM_OUT}/{name}__depth_anything_v2.png').convert('RGB').resize((1024, 768))
    for i, style in enumerate(STYLES):
        img = pipe(prompt=style + ', interior photo, realistic, high quality', negative_prompt='blurry, distorted, text, watermark',
                   image=dimg, controlnet_conditioning_scale=0.7, num_inference_steps=30, generator=torch.manual_seed(1)).images[0]
        path = f'{ROOM_OUT}/{name}__redesign_{i + 1}.png'; img.save(path); made.append((path, style)); print(path)
del pipe, controlnet; gc.collect(); torch.cuda.empty_cache()
# CLIP: does each picture match the style it was asked for? (higher = better)
from transformers import CLIPModel, CLIPProcessor
cm = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').cuda(); cp = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
for path, style in made:
    inp = cp(text=STYLES, images=Image.open(path), return_tensors='pt', padding=True).to('cuda')
    p = cm(**inp).logits_per_image.softmax(-1)[0].tolist()
    print(os.path.basename(path), 'style match %.0f%%' % (100 * p[STYLES.index(style)]))

## Step 9: zip the room outputs

In [ ]:
import zipfile
z = f'{OUT}/zips/room_outputs.zip'; os.makedirs(f'{OUT}/zips', exist_ok=True)
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(ROOM_OUT)): zf.write(f'{ROOM_OUT}/{f}', f)
print(os.path.getsize(z) // 1_000_000, 'MB')
files.download(z)

## Back on your PC
Each model zip is imported on its own. For a model you want to use (replace `triposr` with the model name):
```
node scripts/colab-batch.mjs import models_triposr.zip --model triposr --replace
node scripts/image-to-3d.mjs --normalise
node scripts/bundle-models.mjs
```
`--model` strips the `__triposr` ending so the file becomes `<product-id>.glb`. Importing a second model with `--replace` keeps the old file as `<id>.old.glb`.
In each `report_<model>.json`, `photoFit` below 0.55 / `review: true` means the shape does not match the catalogue photo: check it before publishing.
`room_outputs.zip` is for inspection and design work; it does not go into the app.